In [10]:
import os
from pdf2image import convert_from_path
from PIL import Image

def pdf_to_img(pdf_path):
    folder = os.path.dirname(pdf_path)
    filename = os.path.splitext(os.path.basename(pdf_path))[0]

    images = convert_from_path(pdf_path)

    images = [img.convert("RGB") for img in images]

    widths, heights = zip(*(img.size for img in images))
    total_height = sum(heights)
    max_width = max(widths)

    merged_image = Image.new("RGB", (max_width, total_height), color=(255, 255, 255))

    y_offset = 0
    for img in images:
        merged_image.paste(img, (0, y_offset))
        y_offset += img.height

    output_path = os.path.join(folder, f"{filename}.jpg")
    merged_image.save(output_path, "JPEG", quality=95)


main = os.getcwd().split("notebooks")[0]
# image_file = '0ab3ad91a7274a6057a9d5c60b62eba2a51b541ce3c08af35517012a3f44eb8bc58d204f819c456fb0e60cb116767d8d.pdf'
image_file = "fv090042726300825000021c9"
image_file = os.path.join(main, "data", "extracted", image_file)


def pdf_to_img(pdf_path):
    folder = os.path.dirname(pdf_path)
    filename = os.path.splitext(os.path.basename(pdf_path))[0]

    images = convert_from_path(pdf_path, dpi=100)

    for i, img in enumerate(images):
        img_filename = f"{filename}_page_{i+1}.jpg"
        img_path = os.path.join(folder, img_filename)

        # Save the image in JPG format
        img.save(img_path, "JPEG")
        print(f"Saved: {img_path}")

pdf_to_img(image_file)

Saved: F:\Documentos\git\automaton_1\data\extracted\fv090042726300825000021c9_page_1.jpg


Para guardar: https://github.com/oomol-flows/deepseek-ocr/blob/main/tasks/deepseek_ocr/__init__.py

In [ ]:
from contextlib import contextmanager
import sys

@contextmanager
def suppress_output():
    import builtins

    _original_print = builtins.print
    stdout_fd = sys.stdout.fileno()
    stderr_fd = sys.stderr.fileno()

    # Duplicate original file descriptors
    stdout_dup = os.dup(stdout_fd)
    stderr_dup = os.dup(stderr_fd)

    # Open /dev/null
    devnull = os.open(os.devnull, os.O_WRONLY)

    try:
        # Override print function
        builtins.print = lambda *args, **kwargs: None

        # Redirect file descriptors to /dev/null
        os.dup2(devnull, stdout_fd)
        os.dup2(devnull, stderr_fd)

        # Flush to ensure all buffered data is written
        sys.stdout.flush()
        sys.stderr.flush()

        yield
    finally:
        # Flush again before restoring
        sys.stdout.flush()
        sys.stderr.flush()

        # Restore file descriptors
        os.dup2(stdout_dup, stdout_fd)
        os.dup2(stderr_dup, stderr_fd)

        # Close duplicates and devnull
        os.close(stdout_dup)
        os.close(stderr_dup)
        os.close(devnull)

        # Restore print function
        builtins.print = _original_print

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch
import os

torch.cuda.empty_cache()


model_name = 'deepseek-ai/DeepSeek-OCR'
os.environ["CUDA_VISIBLE_DEVICES"] = '0'
os.environ["FLASH_ATTENTION_TRITON_AMD_ENABLE"] = "TRUE"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(model_name,
                                  # _attn_implementation='flash_attention_2',
                                  trust_remote_code=True, use_safetensors=True)
model = model.eval().cuda().to(torch.bfloat16)



# prompt = "<image>\nFree OCR. "
main = os.getcwd().split("notebooks")[0]
# prompt = "<image>\n"
prompt = "\n<|grounding|>Convert the document to markdown."

image_file = '0ab3ad91a7274a6057a9d5c60b62eba2a51b541ce3c08af35517012a3f44eb8bc58d204f819c456fb0e60cb116767d8d_page_2.jpg'
image_file = os.path.join(main, "data", "extracted", image_file)
output_path = os.path.join(main, "data", "deepseek")

if not os.path.exists(image_file):
    print("Image file does not exist:", image_file)

# infer(self, tokenizer, prompt='', image_file='', output_path = ' ', base_size = 1024, image_size = 640, crop_mode = True, test_compress = False, save_results = False):

# Tiny: base_size = 512, image_size = 512, crop_mode = False
# Small: base_size = 640, image_size = 640, crop_mode = False
# Base: base_size = 1024, image_size = 1024, crop_mode = False
# Large: base_size = 1280, image_size = 1280, crop_mode = False
# Gundam: base_size = 1024, image_size = 640, crop_mode = True
results = model.infer(tokenizer,
            prompt=prompt,
            image_file=image_file,
            base_size=1024,
            image_size=1024,
            crop_mode=False,
            test_compress=False)

In [1]:
from PIL import Image
from transformers import AutoTokenizer, AutoProcessor, AutoModelForImageTextToText
import os

model_path = "nanonets/Nanonets-OCR-s"

os.environ["FLASH_ATTENTION_TRITON_AMD_ENABLE"] = "TRUE"

model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    torch_dtype="auto",
    device_map="auto",
    # attn_implementation="flash_attention_2"
)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(model_path)
processor = AutoProcessor.from_pretrained(model_path)

def ocr_page_with_nanonets_s(image_path, model, processor, max_new_tokens=4096):
    prompt = """Extract the text from the above document as if you were reading it naturally. Return the tables in html format. Return the equations in LaTeX representation. If there is an image in the document and image caption is not present, add a small description of the image inside the <img></img> tag; otherwise, add the image caption inside <img></img>. Watermarks should be wrapped in brackets. Ex: <watermark>OFFICIAL COPY</watermark>. Page numbers should be wrapped in brackets. Ex: <page_number>14</page_number> or <page_number>9/22</page_number>. Prefer using ☐ and ☑ for check boxes."""
    image = Image.open(image_path)
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": [
            {"type": "image", "image": f"file://{image_path}"},
            {"type": "text", "text": prompt},
        ]},
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], padding=True, return_tensors="pt")
    inputs = inputs.to(model.device)

    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, output_ids)]

    output_text = processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)
    return output_text[0]


main = os.getcwd().split("notebooks")[0]
image_file = '0ab3ad91a7274a6057a9d5c60b62eba2a51b541ce3c08af35517012a3f44eb8bc58d204f819c456fb0e60cb116767d8d_page_2.jpg'
image_file = os.path.join(main, "data", "extracted", image_file)
result = ocr_page_with_nanonets_s(image_file, model, processor, max_new_tokens=15000)
print(result)

`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

LookupError: <ContextVar name='shell_parent' at 0x000001E691C70590>

### Mineru 2.5

In [ ]:
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from PIL import Image
from mineru_vl_utils import MinerUClient
import os

model = Qwen2VLForConditionalGeneration.from_pretrained(
    "opendatalab/MinerU2.5-2509-1.2B",
    dtype="auto", # use `torch_dtype` instead of `dtype` for transformers<4.56.0
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(
    "opendatalab/MinerU2.5-2509-1.2B",
    use_fast=True
)

client = MinerUClient(
    backend="transformers",
    model=model,
    processor=processor
)

main = os.getcwd().split("notebooks")[0]
image_file = '0ab3ad91a7274a6057a9d5c60b62eba2a51b541ce3c08af35517012a3f44eb8bc58d204f819c456fb0e60cb116767d8d_page_2.jpg'
image_file = os.path.join(main, "data", "extracted", image_file)

image = Image.open(image_file)
extracted_blocks = client.two_step_extract(image)
print(extracted_blocks)

In [11]:
only_text = ""
for block in extracted_blocks:
    if not "content" in block:
        continue
    text = block["content"]
    if text is None:
        continue
    only_text += text + "\n"

In [13]:
text_2 = client.two_step_extract(image)

Predict: 100%|██████████| 69/69 [00:44<00:00,  1.54it/s]


In [ ]:
from transformers import AutoProcessor
from transformers import HunYuanVLForConditionalGeneration
from PIL import Image
import torch
import os

def clean_repeated_substrings(text):
    """Clean repeated substrings in text"""
    n = len(text)
    if n<8000:
        return text
    for length in range(2, n // 10 + 1):
        candidate = text[-length:]
        count = 0
        i = n - length

        while i >= 0 and text[i:i + length] == candidate:
            count += 1
            i -= length

        if count >= 10:
            return text[:n - length * (count - 1)]

    return text

model_name_or_path = "tencent/HunyuanOCR"
processor = AutoProcessor.from_pretrained(model_name_or_path, use_fast=False)

main = os.getcwd().split("notebooks")[0]
image_file = '0ab3ad91a7274a6057a9d5c60b62eba2a51b541ce3c08af35517012a3f44eb8bc58d204f819c456fb0e60cb116767d8d_page_2.jpg'
image_file = os.path.join(main, "data", "extracted", image_file)

image_inputs = Image.open(image_file)

recomended_prompt = """
• Identify the formula in the image and represent it using LaTeX format.

• Parse the table in the image into HTML.

• Parse the chart in the image; use Mermaid format for flowcharts and Markdown for other charts.

• Extract all information from the main body of the document image and represent it in markdown format, ignoring headers and footers. Tables should be expressed in HTML format, formulas in the document should be represented using LaTeX format, and the parsing should be organized according to the reading order.

"""

messages1 = [
    {"role": "system", "content": ""},
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image_file},
            {"type": "text", "text": (
                recomended_prompt
            )},
        ],
    }
]
messages = [messages1]
texts = [
    processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    for msg in messages
]
inputs = processor(
    text=texts,
    images=image_inputs,
    padding=True,
    return_tensors="pt",
)
model = HunYuanVLForConditionalGeneration.from_pretrained(
    model_name_or_path,
    attn_implementation="eager",
    dtype=torch.bfloat16,
    device_map="auto"
)
with torch.no_grad():
    device = next(model.parameters()).device
    inputs = inputs.to(device)
    generated_ids = model.generate(**inputs, max_new_tokens=16384, do_sample=False)
if "input_ids" in inputs:
    input_ids = inputs.input_ids
else:
    print("inputs: # fallback", inputs)
    input_ids = inputs.inputs
generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(input_ids, generated_ids)
]

raw_output_texts = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(type(raw_output_texts))
out_text = ""

for t in raw_output_texts:
    out_text += clean_repeated_substrings(t) + "\n"

output_texts = clean_repeated_substrings(raw_output_texts[0])
print(output_texts)
print(out_text)

In [ ]:
import subprocess
import os

main = os.getcwd().split("notebooks")[0]
image_file = '0ab3ad91a7274a6057a9d5c60b62eba2a51b541ce3c08af35517012a3f44eb8bc58d204f819c456fb0e60cb116767d8d_page_2.jpg'
image_file = os.path.join(main, "data", "extracted", image_file)

prompt = "<|grounding|>Convert the document to markdown."

full_prompt = f"{image_file}\n{prompt}"

process = subprocess.run(
    ["ollama", "run", "deepseek-ocr"],
    input=full_prompt,
    text=True,
    capture_output=True,
    encoding="utf-8"
)

print(process)